In [1]:
# ============================================================
# CELL 1: Pakete installieren (einmalig im Anaconda Prompt)
# pip install git+https://github.com/apple/ml-mobileclip.git
# ============================================================
import mobileclip
print('✅ mobileclip importiert')

C:\Users\Lukas\miniconda3\envs\kilimanjaro\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lukas\miniconda3\envs\kilimanjaro\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


✅ mobileclip importiert


In [2]:
# ============================================================
# CELL 2: Gewichte herunterladen
# Einmalig ausführen – speichert in Downloads
# ============================================================
import urllib.request
import os

weights_dir  = r"C:\Users\Lukas\Downloads\mobileclip_weights"
weights_path = os.path.join(weights_dir, "mobileclip_s1.pt")
os.makedirs(weights_dir, exist_ok=True)

if not os.path.exists(weights_path):
    print("Lade MobileCLIP-S1 Gewichte herunter (~100MB)...")
    urllib.request.urlretrieve(
        "https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s1.pt",
        weights_path
    )
    print(f"✅ Gewichte gespeichert: {weights_path}")
else:
    print(f"✅ Gewichte bereits vorhanden: {weights_path}")

Lade MobileCLIP-S1 Gewichte herunter (~100MB)...
✅ Gewichte gespeichert: C:\Users\Lukas\Downloads\mobileclip_weights\mobileclip_s1.pt


In [3]:
# ============================================================
# CELL 3: Imports + Modell laden
# ============================================================
import os
import shutil
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
import mobileclip

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

weights_path = r"C:\Users\Lukas\Downloads\mobileclip_weights\mobileclip_s1.pt"

model, _, preprocess = mobileclip.create_model_and_transforms(
    'mobileclip_s1',
    pretrained=weights_path,
    device=device
)
model.eval()
print("Modell geladen: MobileCLIP-S1")

Gerät: cpu
Modell geladen: MobileCLIP-S1


In [ ]:
# ============================================================
# CELL 4: Prompts
# ============================================================
categories = ["indoor", "outdoor"]

category_texts = [
    "a photo taken inside a room showing walls, ceiling, furniture, artificial lighting or floor",
    "a photo taken outdoors showing open spaces such as nature, sky, streets, buildings, parks or landscapes"
]

tokenizer   = mobileclip.get_tokenizer('mobileclip_s1')
text_tokens = tokenizer(category_texts).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("Kategorien:", categories)
print("Text-Embeddings berechnet:", text_features.shape)

In [ ]:
# ============================================================
# CELL 5: Pfade
# ============================================================
image_folder = r"C:\Users\Lukas\Downloads\flickr-resnet50-aestheticNCP\data\flickr_images\vitl14_sorted"

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_paths = []

for subfolder in ['indoor', 'outdoor']:
    folder = os.path.join(image_folder, subfolder)
    if not os.path.exists(folder):
        print(f'⚠ Ordner nicht gefunden: {folder}')
        continue
    for fname in sorted(os.listdir(folder)):
        if fname.startswith("augmented_"):
            continue
        ext = os.path.splitext(fname)[1].lower()
        if ext not in VALID_EXTENSIONS:
            continue
        fpath = os.path.join(folder, fname)
        if os.path.isfile(fpath):
            image_paths.append(fpath)

print(f"Bilder gefunden: {len(image_paths)}")

In [ ]:
# ============================================================
# CELL 6: Klassifikation
# ============================================================
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        print(f"  ⚠ Skipped ({os.path.basename(path)}): {e}")
        return None

BATCH_SIZE = 32

results     = []
confidences = []
valid_paths = []

for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Batches"):
    batch_paths = image_paths[i : i + BATCH_SIZE]
    loaded      = [(p, safe_open(p)) for p in batch_paths]
    loaded      = [(p, img) for p, img in loaded if img is not None]

    if not loaded:
        continue

    paths_ok, imgs_ok = zip(*loaded)

    img_tensors = torch.stack([preprocess(img) for img in imgs_ok]).to(device)

    with torch.no_grad():
        image_features = model.encode_image(img_tensors)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    similarities = image_features @ text_features.T
    batch_preds  = similarities.argmax(dim=1).cpu().numpy()
    batch_conf   = similarities.max(dim=1).values.cpu().float().numpy()

    results.extend([categories[idx] for idx in batch_preds])
    confidences.extend(batch_conf)
    valid_paths.extend(paths_ok)

print(f"\nKlassifiziert: {len(results)} Bilder")
print("\nVorhersage-Verteilung:")
for cat in categories:
    print(f"  {cat}: {results.count(cat)}")

In [ ]:
# ============================================================
# CELL 7: Bilder sortieren
# ============================================================
output_base = r"C:\Users\Lukas\Downloads\flickr-resnet50-aestheticNCP\data\flickr_images\mobileclip_sorted"

sorted_dirs = {cat: os.path.join(output_base, cat) for cat in categories}
for d in sorted_dirs.values():
    os.makedirs(d, exist_ok=True)

for path, pred in zip(valid_paths, results):
    shutil.copy(path, sorted_dirs[pred])

print("Bilder sortiert:")
for cat in categories:
    print(f"  {cat}: {results.count(cat)} → {sorted_dirs[cat]}")

In [ ]:
# ============================================================
# CELL 8: Konfidenz visualisieren
# ============================================================
plt.figure(figsize=(8, 4))
plt.hist(confidences, bins=30, alpha=0.7, color="steelblue")
plt.xlabel("Cosine Similarity")
plt.ylabel("Number of Images")
plt.title("Confidence Distribution – MobileCLIP-S1")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 9: CSV speichern
# ============================================================
output_csv = r"C:\Users\Lukas\Downloads\flickr-resnet50-aestheticNCP\data\flickr_images\mobileclip_sorted\mobileclip_labels.csv"

df_results = pd.DataFrame({
    "file_name":  [os.path.basename(p) for p in valid_paths],
    "pred_label": results,
    "confidence": confidences
})

df_results.to_csv(output_csv, index=False)
print(f"Labels gespeichert: {output_csv}")
print()
print("Verteilung:")
print(df_results["pred_label"].value_counts())
print(f"\nGesamt: {len(df_results)} Bilder")

In [ ]:
# ============================================================
# CELL 10: Unsicherste Bilder
# ============================================================
df_uncertain = df_results.nsmallest(10, "confidence")
print(df_uncertain)

In [ ]:
# ============================================================
# CELL 11: Evaluation mit Ground Truth Labels
# ============================================================
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

output_dir = r"C:\Users\Lukas\Downloads\flickr-resnet50-aestheticNCP\data\flickr_images\mobileclip_sorted\evaluation"
os.makedirs(output_dir, exist_ok=True)

label_csv   = r"C:\Users\Lukas\Downloads\flickr-resnet50-aestheticNCP\data\flickr_images\vitl14_sorted\photo_labels_indoor_outdoor.csv"
df_labels   = pd.read_csv(label_csv)
labels_dict = dict(zip(df_labels["file_name"], df_labels["Main_focus"]))

df_results["true_label"] = df_results["file_name"].map(labels_dict)
df_eval = df_results.dropna(subset=["true_label"])
print(f"Evaluierbare Bilder: {len(df_eval)}")

true     = df_eval["true_label"].values
pred     = df_eval["pred_label"].values
accuracy = accuracy_score(true, pred)
avg_conf = float(np.mean(df_eval["confidence"]))

print(f"\nAccuracy:              {accuracy:.4f}")
print(f"Ø Confidence (Cosine): {avg_conf:.4f}")
print()
print(classification_report(true, pred, target_names=sorted(categories)))

pd.DataFrame(
    classification_report(true, pred,
                          target_names=sorted(categories),
                          output_dict=True)
).transpose().to_csv(os.path.join(output_dir, "mobileclip_metrics.csv"))

le = LabelEncoder()
le.classes_ = np.array(sorted(categories))
cm = confusion_matrix(le.transform(true), le.transform(pred))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix – MobileCLIP-S1 (Accuracy: {accuracy:.2%})")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "mobileclip_confusion_matrix.png"),
            dpi=150, bbox_inches='tight')
plt.show()

correct      = (df_eval["true_label"] == df_eval["pred_label"]).tolist()
conf_correct = [c for c, ok in zip(df_eval["confidence"], correct) if ok]
conf_wrong   = [c for c, ok in zip(df_eval["confidence"], correct) if not ok]

plt.figure(figsize=(8, 4))
plt.hist(conf_correct, bins=30, alpha=0.7,
         label=f"Correct (n={len(conf_correct)})", color="steelblue")
plt.hist(conf_wrong,   bins=30, alpha=0.7,
         label=f"Wrong   (n={len(conf_wrong)})",   color="tomato")
plt.xlabel("Cosine Similarity")
plt.ylabel("Number of Images")
plt.title("Confidence: Correct vs. Wrong – MobileCLIP-S1")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "mobileclip_confidence_distribution.png"),
            dpi=150, bbox_inches='tight')
plt.show()

df_uncertain = df_eval.nsmallest(10, "confidence")[
    ["file_name", "true_label", "pred_label", "confidence"]]
print("\nUnsicherste 10 Bilder:")
print(df_uncertain)
df_uncertain.to_csv(os.path.join(output_dir, "mobileclip_most_uncertain.csv"), index=False)

print(f"\n── Alle Dateien gespeichert in: {output_dir} ──")

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

False


AssertionError: Torch not compiled with CUDA enabled